# Introduction - Customer Churn Prediction notebook
In this notebook, we illustrate how you can train a model for Churn Prediction using PySpark. After training the model, you step through the instructions to deploy the model using Watson Machine Learning.

## Package installation

In [1]:
try:
    from pyspark.sql import SparkSession
except:
    print('Error: Spark runtime is missing. If you are using Watson Studio change the notebook runtime to Spark.')
    raise 
    
!pip list | grep ibm-watson-openscale     

ibm-watson-openscale          3.0.37


In [2]:
# install required Python modules

!pip install --upgrade ibm-watsonx-ai --user | tail -n 1
# !pip install --upgrade "ibm-watson-openscale~=3.0.34" --no-cache --user | tail -n 1


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


# Model building and deployment <a name="model"></a>

In this section you will learn how to train Spark MLLib model and next deploy it as web-service using Watson Machine Learning service.

## Load the training data 

- Click in the next cell to insert the code to import the training dataset.
- Click the **Code snippet </>** icon in the top right, find the data set you'd like to import (for example, CUSTOMER_DATA_ready or customer_training_data.csv) into this notebook and click **Insert to code** drop down and select **pandas DataFrame**

In [2]:
import itc_utils.flight_service as itcfs

nb_data_request = {
    'data_name': """CUSTOMER_DATA_ready""",
    'interaction_properties': {
        #'row_limit': 500,
        'infer_schema': 'true',
        'infer_as_varchar': 'false'
    }
}
flight_descriptor = itcfs.get_flight_descriptor(nb_data_request=nb_data_request)

flightClient = itcfs.get_flight_client()
flightInfo = flightClient.get_flight_info(flight_descriptor)

df_0 = itcfs.read_pandas_and_concat(flightClient, flightInfo, timeout=240)
df_0.head(10)


,ID,LONGDISTANCE,INTERNATIONAL,LOCAL,DROPPED,PAYMETHOD,LOCALBILLTYPE,LONGDISTANCEBILLTYPE,USAGE,RATEPLAN,GENDER,STATUS,CHILDREN,ESTINCOME,CAROWNER,AGE,CHURN
0,1018,21,0,87,0,CC,Budget,Standard,108,1,F,S,0,95786.8,Y,52.646667,F
1,110,1,0,11,0,CC,Budget,Standard,14,3,F,S,2,21021.6,Y,62.753333,T
2,1134,2,0,11,0,CC,Budget,Standard,14,2,M,S,2,21021.6,Y,62.753333,F
3,120,29,0,52,0,Auto,FreeLocal,Standard,82,2,F,M,2,62061.8,Y,45.920000,F
4,1233,20,0,23,0,CC,Budget,Intnl_discount,43,3,F,S,2,76791.2,N,43.000000,F
5,1334,22,8,75,0,CH,Budget,Intnl_discount,106,2,F,S,0,38000.0,N,22.433333,T
6,1491,2,0,49,0,Auto,FreeLocal,Standard,50,2,M,S,0,80221.2,N,29.026667,F
7,1496,12,0,18,0,CC,Budget,Standard,31,1,F,M,0,59008.4,Y,30.180000,T
8,1530,21,0,87,0,CC,Budget,Standard,108,4,F,S,0,95786.8,Y,46.000000,F
9,1915,24,5,60,0,CC,FreeLocal,Standard,90,1,F,M,2,68462.8,N,34.400000,F


In [3]:
# Create a PySpark DataFrame from the pandas DataFrame
from pyspark.sql import SparkSession
import pandas as pd

import json
# Provide the name of the pandas DataFrame from the previous cell (should be of the format df_data_<some_number>)
pandasDFname=df_0
spark = SparkSession.builder.getOrCreate()
sparkDF=spark.createDataFrame(pandasDFname)
sparkDF.head()

Row(ID=1018, LONGDISTANCE=21, INTERNATIONAL=0, LOCAL=87, DROPPED=0, PAYMETHOD='CC', LOCALBILLTYPE='Budget', LONGDISTANCEBILLTYPE='Standard', USAGE=108, RATEPLAN=1, GENDER='F', STATUS='S', CHILDREN=0, ESTINCOME=95786.8, CAROWNER='Y', AGE=52.646667, CHURN='F')

## Explore data

In [4]:
sparkDF.printSchema()

root
 |-- ID: long (nullable = true)
 |-- LONGDISTANCE: long (nullable = true)
 |-- INTERNATIONAL: long (nullable = true)
 |-- LOCAL: long (nullable = true)
 |-- DROPPED: long (nullable = true)
 |-- PAYMETHOD: string (nullable = true)
 |-- LOCALBILLTYPE: string (nullable = true)
 |-- LONGDISTANCEBILLTYPE: string (nullable = true)
 |-- USAGE: long (nullable = true)
 |-- RATEPLAN: long (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- CHILDREN: long (nullable = true)
 |-- ESTINCOME: double (nullable = true)
 |-- CAROWNER: string (nullable = true)
 |-- AGE: double (nullable = true)
 |-- CHURN: string (nullable = true)



In [5]:
print("Number of records: " + str(sparkDF.count()))

Number of records: 1415


## Preprocessing

In [6]:
from pyspark.sql.functions import count, when, col, sum
print(sparkDF.select([count(when(col(c).isNull(), c)).alias(c) for c in sparkDF.columns]))
has_nulls = sparkDF.select([count(when(col(c).isNull(), c)).alias(c) for c in sparkDF.columns]).collect()[0]
if any(value > 0 for value in has_nulls.asDict().values()):
    print("DataFrame contains null values.")
else:
    print("DataFrame does not contain null values.")


DataFrame[ID: bigint, LONGDISTANCE: bigint, INTERNATIONAL: bigint, LOCAL: bigint, DROPPED: bigint, PAYMETHOD: bigint, LOCALBILLTYPE: bigint, LONGDISTANCEBILLTYPE: bigint, USAGE: bigint, RATEPLAN: bigint, GENDER: bigint, STATUS: bigint, CHILDREN: bigint, ESTINCOME: bigint, CAROWNER: bigint, AGE: bigint, CHURN: bigint]
DataFrame does not contain null values.


In [7]:
# Check for missing values
sparkDF.select(*(sum(col(c).isNull().cast("int")).alias(c) for c in sparkDF.columns)).show()

+---+------------+-------------+-----+-------+---------+-------------+--------------------+-----+--------+------+------+--------+---------+--------+---+-----+
| ID|LONGDISTANCE|INTERNATIONAL|LOCAL|DROPPED|PAYMETHOD|LOCALBILLTYPE|LONGDISTANCEBILLTYPE|USAGE|RATEPLAN|GENDER|STATUS|CHILDREN|ESTINCOME|CAROWNER|AGE|CHURN|
+---+------------+-------------+-----+-------+---------+-------------+--------------------+-----+--------+------+------+--------+---------+--------+---+-----+
|  0|           0|            0|    0|      0|        0|            0|                   0|    0|       0|     0|     0|       0|        0|       0|  0|    0|
+---+------------+-------------+-----+-------+---------+-------------+--------------------+-----+--------+------+------+--------+---------+--------+---+-----+



In [8]:
duplicate_count = sparkDF.groupBy(sparkDF.columns).count().filter("count > 1").count()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


## Create a model

In [9]:
spark_df = sparkDF
# Split the labeled data into a training set and a test set
(train_data, test_data) = spark_df.randomSplit([0.8, 0.2], 24)

# Provide a target name for your churn model
MODEL_NAME = "Churn Model"
# Provide a target name for your churn model deployment
DEPLOYMENT_NAME = "Churn Deployment"

print("Number of records for training: " + str(train_data.count()))
print("Number of records for evaluation: " + str(test_data.count()))

Number of records for training: 1158
Number of records for evaluation: 257


The code below creates a Random Forest Classifier with Spark, setting up string indexers for the categorical features and the label column. Finally, this notebook creates a pipeline including the indexers and the model, and does an initial Area Under ROC evaluation of the model.

In [10]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer, IndexToString, VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline, Model
from pyspark.ml.feature import SQLTransformer

features = [x for x in spark_df.columns if x != 'CHURN']
# Specify the categorical features
categorical_features = ['PAYMETHOD', 'LOCALBILLTYPE', 'LONGDISTANCEBILLTYPE', 'GENDER', 'STATUS', 'CAROWNER']
# Index the categorical feature so each string value is replaced with an integer
categorical_num_features = [x + '_IX' for x in categorical_features]
si_list = [StringIndexer(inputCol=x, outputCol=y) for x, y in zip(categorical_features, categorical_num_features)]
va_features = VectorAssembler(inputCols=categorical_num_features + [x for x in features if x not in categorical_features], outputCol="features")

In [11]:
# Index the label column
si_label = StringIndexer(inputCol="CHURN", outputCol="label").fit(spark_df)
label_converter = IndexToString(inputCol="prediction", outputCol="predictedLabel", labels=si_label.labels)

In [12]:
from pyspark.ml.classification import RandomForestClassifier
# train a Random Forect Classifier
classifier = RandomForestClassifier(featuresCol="features")
pipeline = Pipeline(stages= si_list + [si_label, va_features, classifier, label_converter])

model = pipeline.fit(train_data)

In [13]:
predictions = model.transform(test_data)
evaluatorDT = BinaryClassificationEvaluator(rawPredictionCol="prediction",  metricName='areaUnderROC')
area_under_curve = evaluatorDT.evaluate(predictions)

evaluatorDT = BinaryClassificationEvaluator(rawPredictionCol="prediction",  metricName='areaUnderPR')
area_under_PR = evaluatorDT.evaluate(predictions)
#default evaluation is areaUnderROC
print("areaUnderROC = %g" % area_under_curve, "areaUnderPR = %g" % area_under_PR)

areaUnderROC = 0.907998 areaUnderPR = 0.849747


In [14]:
# extra code: evaluate more metrics by exporting them into pandas and numpy
from sklearn.metrics import classification_report
y_pred = predictions.toPandas()['prediction']
y_pred = ['T' if pred == 1.0 else 'F' for pred in y_pred]
y_test = test_data.toPandas()['CHURN']
print(classification_report(y_test, y_pred, target_names=['T', 'F']))

              precision    recall  f1-score   support

           T       0.94      0.90      0.92       149
           F       0.87      0.92      0.89       108

    accuracy                           0.91       257
   macro avg       0.90      0.91      0.90       257
weighted avg       0.91      0.91      0.91       257



## watsonx.ai connection

Authenticate the watsonx.ai Runtime service on IBM Cloud. You need to provide platform api_key and instance location.

You can find your api_key by clicking the **icon at top right > Profile and Settings > API Key > Generate a new key.**

In [15]:
import getpass

api_key = getpass.getpass("Please enter your api key (press enter): ")

Please enter your api key (press enter):  ········


In [16]:
credentials = {
    'username': 'cpadmin',
    'apikey': api_key,
    'url' : 'https://cpd-cpd-instance.apps.691c390807fea3b75366233a.am1.techzone.ibm.com/',
    'instance_id': 'openshift'
}

In [17]:
print(credentials)

{'username': 'cpadmin', 'apikey': 'YA7hNpcPOIZkaQxeeroVlAuGp6aiSwnsLhs66aD8', 'url': 'https://cpd-cpd-instance.apps.691c390807fea3b75366233a.am1.techzone.ibm.com/', 'instance_id': 'openshift'}


In [18]:
from ibm_watsonx_ai import APIClient
client = APIClient(credentials)

In [19]:
# List existing deployment space
client.spaces.list(limit=10)

,ID,NAME,CREATED
0,4ec09120-3d15-430d-8d3e-8ffb6023ab86,Test,2025-11-25T09:28:03.980Z


## Publish the model

In [20]:
def getSpaceIDwml(wml_client,space_name):
    spaces = wml_client.spaces.get_details()['resources'];
    try:
        spaceList = next(item for item in spaces if item['entity']['name']==space_name)
        spaceID = spaceList['metadata']['id']
    except:
        spaceID = -1
    return spaceID

In [23]:
import time
def createSpacewml(wml_client,space_name):
    spaces = wml_client.spaces.get_details()['resources'];
    for space in spaces:
        if space['entity']['name'] ==space_name:
            print("Deployment space with name",space_name,"already exists . .")
            return space['metadata']['id']
    print("\nCreating a new deployment space -",space_name)
    # create the space
    space_meta_data = {
        wml_client.spaces.ConfigurationMetaNames.NAME : space_name
    }

    stored_space_details = wml_client.spaces.store(space_meta_data)
    space_id = stored_space_details['metadata']['id']
    i=0
    while(True):
        stored_space_details=wml_client.spaces.get_details(space_id)
        status=stored_space_details['entity']['status']['state']
        print("i: ", i, " status: ", status)
        if status == 'active':
            break
        time.sleep(1)
        i = i+1
    return space_id

In [25]:
# Associate Watson Machine Learning with a specific space

space_name = 'churnUATSpace' #Replace any name of choice here
space_id=getSpaceIDwml(client,space_name)
if space_id == -1:
    space_id = createSpacewml(client,space_name)
print('space id: ', space_id)
client.set.default_space(space_id)



Creating a new deployment space - churnUATSpace
Space has been created. However some background setup activities might still be on-going. Check for 'status' field in the response. It has to show 'active' before space can be used. If it's not 'active', you can monitor the state with a call to spaces.get_details(space_id). Alternatively, use background_mode=False when calling client.spaces.store().
i:  0  status:  preparing
i:  1  status:  active
space id:  77db322f-6b97-46e3-8563-96dc06939e12


'SUCCESS'

In [26]:
def deleteExistingModelsSameName(client,model_name):
    stored_models=client.repository.get_model_details()
    stored_models_details = stored_models['resources']
    for m in stored_models_details:
        m_name = m['metadata']['name']
        if m_name == model_name:
            model_id = m['metadata']['id']
            print("Deleteing model with id: ", model_id, " and name: ", m_name)
            client.repository.delete(model_id)
    return 'Success'

In [27]:
def deleteExistingDeploymentsSameName(client,deployment_name):
    stored_deployments=client.deployments.get_details()
    stored_deployment_details = stored_deployments['resources']
    for d in stored_deployment_details:
        d_name = d['metadata']['name']
        if d_name == deployment_name:
            deployment_id = d['metadata']['id']
            print("Deleteing deployment with id: ", deployment_id, " and name: ", d_name)
            client.deployments.delete(deployment_id)
    return 'Success'

Previous versions of the model are removed so that the notebook can be run again, resetting all data for another demo.

In [28]:
# Delete Existing Deployments with same name
deleteExistingDeploymentsSameName(client,DEPLOYMENT_NAME)

'Success'

In [29]:
# Delete existing models with same name
deleteExistingModelsSameName(client,MODEL_NAME)

'Success'

In [30]:
software_spec_uid = client.software_specifications.get_id_by_name("spark-mllib_3.5")
print("Software Specification ID: {}".format(software_spec_uid))
model_props = {
        client._models.ConfigurationMetaNames.NAME:"{}".format(MODEL_NAME),
        client._models.ConfigurationMetaNames.TYPE: "mllib_3.5",
        client._models.ConfigurationMetaNames.SOFTWARE_SPEC_UID: software_spec_uid,
        #wml_client._models.ConfigurationMetaNames.TRAINING_DATA_REFERENCES: training_data_references,
        client._models.ConfigurationMetaNames.LABEL_FIELD: "CHURN",
    }

Software Specification ID: e8cd7001-fd05-5eea-b697-0a3bff9cf51f


In [31]:
print("Storing model ...")
published_model_details = client.repository.store_model(
    model=model, 
    meta_props=model_props, 
    training_data=train_data, 
    pipeline=pipeline)

model_uid = client.repository.get_model_id(published_model_details)
print("Done")
print("Model ID: {}".format(model_uid))

Storing model ...
Done
Model ID: c3f13fe6-7351-425c-a1bf-b48882b8937b


In [32]:
client.repository.list_models()

,ID,NAME,CREATED,TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,c3f13fe6-7351-425c-a1bf-b48882b8937b,Churn Model,2025-12-16T10:36:28Z,mllib_3.5,supported,


## Deploy the model

The next section of the notebook deploys the model as a RESTful web service in Watson Machine Learning. The deployed model will have a scoring URL you can use to send data to the model for predictions.

In [33]:
deployment_details = client.deployments.create(
    model_uid, 
    meta_props={
        client.deployments.ConfigurationMetaNames.NAME: "{}".format(DEPLOYMENT_NAME),
        client.deployments.ConfigurationMetaNames.ONLINE: {}
    }
)




######################################################################################

Synchronous deployment creation for id: 'c3f13fe6-7351-425c-a1bf-b48882b8937b' started

######################################################################################


initializing
Note: online_url is deprecated and will be removed in a future release. Use serving_urls instead.
..
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='cc1cdea2-290f-47ff-b5e9-afb8c43d6a28'
-----------------------------------------------------------------------------------------------




In [34]:
scoring_url = client.deployments.get_scoring_href(deployment_details)
deployment_uid=client.deployments.get_uid(deployment_details)

print("Scoring URL:" + scoring_url)
print("Model id: {}".format(model_uid))
print("Deployment id: {}".format(deployment_uid))

Scoring URL:https://cpd-cpd-instance.apps.691c390807fea3b75366233a.am1.techzone.ibm.com/ml/v4/deployments/cc1cdea2-290f-47ff-b5e9-afb8c43d6a28/predictions
Model id: c3f13fe6-7351-425c-a1bf-b48882b8937b
Deployment id: cc1cdea2-290f-47ff-b5e9-afb8c43d6a28


In [35]:
client.deployments.list()

,ID,NAME,STATE,CREATED,ARTIFACT_TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,cc1cdea2-290f-47ff-b5e9-afb8c43d6a28,Churn Deployment,ready,2025-12-16T10:36:55.713Z,model,supported,


## Sample scoring

In [36]:
fields = ["ID","LONGDISTANCE","INTERNATIONAL","LOCAL","DROPPED","PAYMETHOD","LOCALBILLTYPE","LONGDISTANCEBILLTYPE","USAGE",\
            "RATEPLAN","GENDER","STATUS","CHILDREN","ESTINCOME","CAROWNER","AGE"]
values = [[1,28,0,60,0,"Auto","FreeLocal","Standard",89,4,"F","M",1,23000,"N",45]]
scoring_payload = {"input_data": [{"fields": fields, "values": values}]}

In [37]:
scoring_response = client.deployments.score(deployment_uid, scoring_payload)
scoring_response

{'predictions': [{'fields': ['ID',
    'LONGDISTANCE',
    'INTERNATIONAL',
    'LOCAL',
    'DROPPED',
    'PAYMETHOD',
    'LOCALBILLTYPE',
    'LONGDISTANCEBILLTYPE',
    'USAGE',
    'RATEPLAN',
    'GENDER',
    'STATUS',
    'CHILDREN',
    'ESTINCOME',
    'CAROWNER',
    'AGE',
    'PAYMETHOD_IX',
    'LOCALBILLTYPE_IX',
    'LONGDISTANCEBILLTYPE_IX',
    'GENDER_IX',
    'STATUS_IX',
    'CAROWNER_IX',
    'label',
    'features',
    'rawPrediction',
    'probability',
    'prediction',
    'predictedLabel'],
   'values': [[1,
     28,
     0,
     60,
     0,
     'Auto',
     'FreeLocal',
     'Standard',
     89,
     4,
     'F',
     'M',
     1,
     23000.0,
     'N',
     45.0,
     1.0,
     1.0,
     0.0,
     0.0,
     0.0,
     0.0,
     0.0,
     [1.0,
      1.0,
      0.0,
      0.0,
      0.0,
      0.0,
      1.0,
      28.0,
      0.0,
      60.0,
      0.0,
      89.0,
      4.0,
      1.0,
      23000.0,
      45.0],
     [15.079787853150414, 4.920212146849